# 02 — Feature Engineering

## Objetivo
Crear nuevas variables a partir de los datos crudos para que los modelos 
de Machine Learning puedan aprender patrones temporales, estacionales 
y del contexto del negocio.

## Features que vamos a construir
1. **Variables temporales** — día de semana, mes, año, trimestre
2. **Variables de lag** — ventas de días anteriores
3. **Variables de rolling** — promedios móviles de ventas
4. **Variables externas** — precio del petróleo e indicador de feriados

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Carga de datos
train = pd.read_csv('../data/raw/train.csv')
oil = pd.read_csv('../data/raw/oil.csv')
holidays = pd.read_csv('../data/raw/holidays_events.csv')

# Convertir fechas
train['date'] = pd.to_datetime(train['date'])
oil['date'] = pd.to_datetime(oil['date'])
holidays['date'] = pd.to_datetime(holidays['date'])

print(f"Train shape: {train.shape}")

Train shape: (3000888, 6)


## 1. Variables Temporales

In [16]:
# Variables temporales extraídas de la fecha
train['dia_semana'] = train['date'].dt.dayofweek      # 0=Lunes, 1=Martes, etc
train['mes'] = train['date'].dt.month                  # 1 a 12
train['año'] = train['date'].dt.year                  # 2013 a 2017
train['trimestre'] = train['date'].dt.quarter          # 1 a 4
train['dia_mes'] = train['date'].dt.day                # 1 a 31
train['semana_año'] = train['date'].dt.isocalendar().week.astype(int)  # 1 a 52
train['es_fin_de_semana'] = (train['dia_semana'] >= 5).astype(int)      # 1=sí, 0=no

print(train[['date', 'dia_semana', 'mes', 'año', 
             'trimestre', 'es_fin_de_semana']].head(7))

        date  dia_semana  mes   año  trimestre  es_fin_de_semana
0 2013-01-01           1    1  2013          1                 0
1 2013-01-01           1    1  2013          1                 0
2 2013-01-01           1    1  2013          1                 0
3 2013-01-01           1    1  2013          1                 0
4 2013-01-01           1    1  2013          1                 0
5 2013-01-01           1    1  2013          1                 0
6 2013-01-01           1    1  2013          1                 0


## 2. Variables de Lag

In [17]:
# Ordenar datos antes de crear lags
train = train.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

# Variables de lag por tienda y familia
train['lag_7'] = train.groupby(['store_nbr', 'family'])['sales'].shift(7)
train['lag_14'] = train.groupby(['store_nbr', 'family'])['sales'].shift(14)
train['lag_28'] = train.groupby(['store_nbr', 'family'])['sales'].shift(28)

print(train[['date', 'store_nbr', 'family', 'sales', 'lag_7', 'lag_14', 'lag_28']].head(10))

        date  store_nbr      family  sales  lag_7  lag_14  lag_28
0 2013-01-01          1  AUTOMOTIVE    0.0    NaN     NaN     NaN
1 2013-01-02          1  AUTOMOTIVE    2.0    NaN     NaN     NaN
2 2013-01-03          1  AUTOMOTIVE    3.0    NaN     NaN     NaN
3 2013-01-04          1  AUTOMOTIVE    3.0    NaN     NaN     NaN
4 2013-01-05          1  AUTOMOTIVE    5.0    NaN     NaN     NaN
5 2013-01-06          1  AUTOMOTIVE    2.0    NaN     NaN     NaN
6 2013-01-07          1  AUTOMOTIVE    0.0    NaN     NaN     NaN
7 2013-01-08          1  AUTOMOTIVE    2.0    0.0     NaN     NaN
8 2013-01-09          1  AUTOMOTIVE    2.0    2.0     NaN     NaN
9 2013-01-10          1  AUTOMOTIVE    2.0    3.0     NaN     NaN


## 3. Variables de Rolling (Promedios Móviles)

In [18]:
# Promedios móviles por tienda y familia
train['rolling_7'] = train.groupby(['store_nbr', 'family'])['sales']\
    .transform(lambda x: x.shift(1).rolling(window=7).mean())

train['rolling_28'] = train.groupby(['store_nbr', 'family'])['sales']\
    .transform(lambda x: x.shift(1).rolling(window=28).mean())

print(train[['date', 'store_nbr', 'family', 'sales', 
             'rolling_7', 'rolling_28']].head(35))

         date  store_nbr      family  sales  rolling_7  rolling_28
0  2013-01-01          1  AUTOMOTIVE    0.0        NaN         NaN
1  2013-01-02          1  AUTOMOTIVE    2.0        NaN         NaN
2  2013-01-03          1  AUTOMOTIVE    3.0        NaN         NaN
3  2013-01-04          1  AUTOMOTIVE    3.0        NaN         NaN
4  2013-01-05          1  AUTOMOTIVE    5.0        NaN         NaN
5  2013-01-06          1  AUTOMOTIVE    2.0        NaN         NaN
6  2013-01-07          1  AUTOMOTIVE    0.0        NaN         NaN
7  2013-01-08          1  AUTOMOTIVE    2.0   2.142857         NaN
8  2013-01-09          1  AUTOMOTIVE    2.0   2.428571         NaN
9  2013-01-10          1  AUTOMOTIVE    2.0   2.428571         NaN
10 2013-01-11          1  AUTOMOTIVE    3.0   2.285714         NaN
11 2013-01-12          1  AUTOMOTIVE    2.0   2.285714         NaN
12 2013-01-13          1  AUTOMOTIVE    2.0   1.857143         NaN
13 2013-01-14          1  AUTOMOTIVE    2.0   1.857143        

### 📝 Interpretación

Los promedios móviles se calcularon correctamente. Las primeras filas muestran NaN 
porque no hay suficientes días anteriores para calcular el promedio — esto es esperado 
y normal. A partir del día 7 aparece `rolling_7` y a partir del día 28 aparece 
`rolling_28`. El `.shift(1)` aplicado antes del cálculo garantiza que no se usa 
información del día actual para predecir ese mismo día, sino de un día anterior. 

## 4. Variables Externas

In [19]:
# Tratamiento del precio del petróleo
oil = oil.set_index('date').reindex(
    pd.date_range(oil['date'].min(), oil['date'].max(), freq='D')
).rename_axis('date').reset_index()

# Interpolación + relleno de bordes
oil['dcoilwtico'] = oil['dcoilwtico'].interpolate(method='linear')
oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill()

# Merge con train
train = train.merge(oil, on='date', how='left')

# Rellenar nulos restantes del merge
train['dcoilwtico'] = train['dcoilwtico'].ffill().bfill()

print(f"Nulos restantes en petróleo: {train['dcoilwtico'].isnull().sum()}")
print(train[['date', 'sales', 'dcoilwtico']].head())

Nulos restantes en petróleo: 0
        date  sales  dcoilwtico
0 2013-01-01    0.0   93.140000
1 2013-01-02    2.0   93.140000
2 2013-01-03    3.0   92.970000
3 2013-01-04    3.0   93.120000
4 2013-01-05    5.0   93.146667


### 📝 Interpretación

El precio del petróleo fue incorporado correctamente al dataset. Los valores nulos 
correspondientes a fines de semana y feriados fueron resueltos mediante interpolación 
lineal, complementada con ffill y bfill para cubrir los extremos de la serie. 
El resultado final es 0 valores nulos en esta variable.

## 5. Variable de Feriados

In [20]:
# Crear indicador de feriado nacional
feriados_nacionales = holidays[holidays['locale'] == 'National']['date'].unique()

train['es_feriado'] = train['date'].isin(feriados_nacionales).astype(int)

print(f"Días con feriado nacional: {train['es_feriado'].sum()}")
print(train[['date', 'sales', 'es_feriado']].head(10))

Días con feriado nacional: 254826
        date  sales  es_feriado
0 2013-01-01    0.0           1
1 2013-01-02    2.0           0
2 2013-01-03    3.0           0
3 2013-01-04    3.0           0
4 2013-01-05    5.0           1
5 2013-01-06    2.0           0
6 2013-01-07    0.0           0
7 2013-01-08    2.0           0
8 2013-01-09    2.0           0
9 2013-01-10    2.0           0


### 📝 Interpretación

Se creó una variable binaria que indica si un día es feriado nacional en Ecuador. 
El dataset contiene 254826 registros correspondientes a feriados nacionales. 
Esta variable es clave para que el modelo aprenda que los feriados tienen un 
comportamiento de ventas distinto al resto de los días.

## 6. Dataset Final

In [22]:
# Resumen del dataset con todos los features
print("=== DATASET FINAL ===")
print(f"Shape: {train.shape}")
print(f"\nColumnas creadas:")
print(train.columns.tolist())
print(f"\nNulos por columna:")
print(train.isnull().sum()[train.isnull().sum() > 0])

# Guardar dataset procesado
train.to_csv('../data/processed/train_features.csv', index=False)

=== DATASET FINAL ===
Shape: (3000888, 20)

Columnas creadas:
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'dia_semana', 'mes', 'año', 'trimestre', 'dia_mes', 'semana_año', 'es_fin_de_semana', 'lag_7', 'lag_14', 'lag_28', 'rolling_7', 'rolling_28', 'dcoilwtico', 'es_feriado']

Nulos por columna:
lag_7         12474
lag_14        24948
lag_28        49896
rolling_7     12474
rolling_28    49896
dtype: int64


### 📝 Interpretación

El dataset final cuenta con 20 columnas — 14 features nuevos creados a partir de 
los 6 originales. Los valores nulos en las variables de lag y rolling son esperados 
y corresponden a los primeros días de cada combinación tienda-familia, donde no 
existe historial suficiente para calcularlos. XGBoost maneja estos nulos 
nativamente sin necesidad de imputación adicional.